In [1]:
!pip install pandas
!pip install pyarrow

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: C:\Users\user\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
  Using cached pyarrow-24.0.0-cp312-cp312-win_amd64.whl.metadata (3.0 kB)
Using cached pyarrow-24.0.0-cp312-cp312-win_amd64.whl (27.4 MB)



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: C:\Users\user\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd

# Загружаем parquet-файл
df = pd.read_parquet("..\\feature_store\\feature_repo\\data\\driver_stats.parquet")

# Смотрим структуру
print("Колонки:", df.columns.tolist())
print("\nТипы данных:")
print(df.dtypes)
print("\nПервые 5 строк:")
print(df.head())
print("\nСтатистика:")
print(df.describe())
print("Минимум:", df["event_timestamp"].min())
print("Максимум:", df["event_timestamp"].max())



Колонки: ['event_timestamp', 'driver_id', 'conv_rate', 'acc_rate', 'avg_daily_trips', 'created']

Типы данных:
event_timestamp    datetime64[ns, UTC]
driver_id                        int64
conv_rate                      float32
acc_rate                       float32
avg_daily_trips                  int32
created                 datetime64[us]
dtype: object

Первые 5 строк:
                   event_timestamp  driver_id  conv_rate  acc_rate  \
0 2024-10-17 12:07:08.228578+00:00       1001   1.000000  1.000000   
1        2024-10-02 11:00:00+00:00       1005   0.429879  0.194598   
2        2024-10-02 12:00:00+00:00       1005   0.230119  0.642878   
3        2024-10-02 13:00:00+00:00       1005   0.128600  0.674187   
4        2024-10-02 14:00:00+00:00       1005   0.400603  0.473636   

   avg_daily_trips                    created  
0             1000 2024-10-17 12:07:08.228581  
1              582 2024-10-17 11:30:07.072000  
2              551 2024-10-17 11:30:07.072000  
3          

Historical data

In [18]:
from feast import FeatureStore
from datetime import datetime
import pandas as pd

store = FeatureStore(repo_path="..\\feature_store\\feature_repo")
store.materialize(
    start_date=datetime(2021, 4, 12, 7, 0, 0),
    end_date=datetime(2024, 10, 17, 12, 7, 8)
)


# Данные для обучения
entity_df = pd.DataFrame.from_dict({
    "driver_id": [1001, 1002, 1003, 1004, 1005],
    "event_timestamp": [datetime(2024, 10, 2)] * 5,
    "target_conv_rate": [0.8, 0.7, 0.9, 0.6, 0.85],
})

# Получаем исторические признаки
historical_features = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "driver_efficiency:conv_rate",
        "driver_efficiency:acc_rate",
        "driver_activity:avg_daily_trips",
        "driver_performance_metrics:performance_score",
        "driver_performance_metrics:is_high_performer",
    ]
)
print()
print('=== Historical Features ===')
print()
print(historical_features.to_df().to_string())



Materializing 2 feature views from 2021-04-12 07:00:00+00:00 to 2024-10-17 12:07:08+00:00 into the sqlite online store.

driver_activity:
driver_efficiency:

=== Historical Features ===

   driver_id           event_timestamp  target_conv_rate  conv_rate  acc_rate  avg_daily_trips  is_high_performer  performance_score
0       1001 2024-10-02 00:00:00+00:00              0.80   0.709758  0.692957              402                  1           1.697790
1       1002 2024-10-02 00:00:00+00:00              0.70   0.718295  0.584081              370                  1           1.572543
2       1003 2024-10-02 00:00:00+00:00              0.90   0.697411  0.197680               25                  0           0.413269
3       1004 2024-10-02 00:00:00+00:00              0.60   0.774095  0.929545              796                  1           2.976502
4       1005 2024-10-02 00:00:00+00:00              0.85   0.186924  0.576559              648                  0           2.191737


In [ ]:
Online Features

In [20]:
from feast import FeatureStore
from datetime import datetime
import pandas as pd

store = FeatureStore(repo_path="../feature_store/feature_repo")
store.materialize(
    start_date=datetime(2021, 4, 12, 7, 0, 0),
    end_date=datetime(2024, 10, 17, 12, 7, 8)
)
# Данные для онлайн-запроса
online_entity_rows = [
    {"driver_id": 1001, "target_conv_rate": 0.85},
    {"driver_id": 1003, "target_conv_rate": 0.90},
]

online_features = store.get_online_features(
    features=[
        "driver_efficiency:conv_rate",
        "driver_efficiency:acc_rate",
        "driver_activity:avg_daily_trips",
        "driver_performance_metrics:efficiency_gap",
        "driver_performance_metrics:performance_score",
        "driver_performance_metrics:is_high_performer",
    ],
    entity_rows=online_entity_rows,
).to_dict()
print()
print('=== Online Features ===')
print(pd.DataFrame(online_features).to_string())

Materializing 2 feature views from 2021-04-12 07:00:00+00:00 to 2024-10-17 12:07:08+00:00 into the sqlite online store.

driver_activity:
driver_efficiency:

=== Online Features ===
   driver_id  conv_rate  acc_rate  avg_daily_trips  efficiency_gap  is_high_performer  performance_score
0       1001   0.295067  0.257097              456        0.554933                  0           1.563156
1       1003   0.934453  0.695821              763       -0.034453                  1           2.871527
